In [3]:
using ITensors
using ITensorMPS

using Random

#=
    Generates the MPO for the EHM Hamiltonian 
    with strengths J, U and V. 
    Requires a SiteType sites.
=#
function H_EHM(N, J, U, V, sites)
    os = OpSum()
    for i in 1:(N - 1)
      # Knetic 
      os -= J, "Cdagup", i, "Cup", i + 1
      os -= J, "Cdagup", i + 1, "Cup", i
      os -= J, "Cdagdn", i, "Cdn", i + 1
      os -= J, "Cdagdn", i + 1, "Cdn", i
      # Nearest-neighbours
      os += V, "Ntot", i, "Ntot", i + 1
    end
    # on-site
    for i in 1:N
      os += U, "Nupdn", i
    end
    return MPO(os, sites)
end


function random_metallic_state(L, Nup, Ndn)
    state = fill("Emp", L)
    p = Nup + Ndn
    for i in 1:L
        j = L - i
        if(p > j)
            state[j] = "UpDn"
            p -= 2
        elseif (p > 0) 
            state[j] = j % 2 == 1 ? "Up" : "Dn"
            p -= 1
        end
    end
    return state
end


using Statistics
function average_single_site_entanglement(L, up, dn, updn)
    single_site_entanglement = fill(0.0, L)

    for i in 1:L 
        w_2 = updn[i] 
        w_up = up[i] - w_2 
        w_dn = dn[i] - w_2 
        w_0 = 1 - w_up - w_dn - w_2 
        single_site_entanglement[i] = 1 - (w_2^2 + w_up^2 + w_dn^2 + w_0^2)
    end
    return Statistics.mean(single_site_entanglement)
end 
#=
    Returns the density of up and down 
    electrons.
=#
function density_operators(N, psi)
    upd = fill(0.0, N)
    dnd = fill(0.0, N)
    updn = fill(0.0, N) 
    for j in 1:N
    orthogonalize!(psi, j)
    psidag_j = dag(prime(psi[j], "Site"))
    upd[j] = scalar(psidag_j * op(sites, "Nup", j) * psi[j])
    dnd[j] = scalar(psidag_j * op(sites, "Ndn", j) * psi[j])
    updn[j] = scalar(psidag_j * op(sites, "Nupdn", j) * psi[j])
    end
    return upd, dnd, updn
end

density_operators (generic function with 1 method)

generating input range of values for $U$ and $V$.

In [6]:
results = "../results/EHM_Itensor_Phase_Diagram/"

U_max = 8.0
V_max = 6.0

U_min = -U_max
V_min = -V_max

N_Points = 50
L = 6

#=
        
        L = 6 e 7 
        N_Points = 100

        L = 5 e 6 
        N_Points = 50

=#
using DelimitedFiles 

U_values = range(U_min, stop=U_max, length=N_Points)

filename = joinpath(results, "U_vals_NPoints=$(N_Points).txt")
writedlm(filename, U_values)

V_values = range(V_min, stop=V_max, length=N_Points)

filename = joinpath(results, "V_vals_NPoints=$(N_Points).txt")
writedlm(filename, V_values)


In [63]:
function average_entanglement_entropy(psi, L)
    # Average entropy taken over all sites. 
    S_avg = 0.0
    # Over all sites j

    for j = 1:L
        # change orthogonality center to j
        psi = orthogonalize(psi, j)
        
        # tensor at site j
        A = psi[j]

        # prime the physical index 
        A_dag = dag(A)
        prime!(A_dag, "Site")

        rdm = A * A_dag

        # Diagonalize
        D, U = eigen(rdm)

        # Compute von Neumann entropy safely
        S = 0.0
        for n=1:dim(D, 1)
            p = D[n,n] 
            p_real = real(p)
            if p_real > 1e-18
                S -= p_real * log(p_real)
            end
        end
        S_avg += S 
    end
    return S_avg / L 
end

average_entanglement_entropy (generic function with 1 method)

In [96]:
L = 7

sites = siteinds("Electron", L; conserve_qns=true)

maxdim = [50, 100, 200, 400, 800, 800]
cutoff = [1E-14]

nsweeps = 10

Npart = floor(Int, L/2) 
Nup = Npart + L % 2 
Ndn = L - Nup 

state = random_metallic_state(L, Nup, Ndn)

println(state)

psi0 = random_mps(sites, state; linkdims=10)

U = 0.5 
V = -7.5
J = 1.0

H = H_EHM(L, J, U, V, sites)

energy, psi = dmrg(H, psi0; nsweeps, maxdim, cutoff)

["Up", "Dn", "Up", "Dn", "Up", "UpDn", "Emp"]
After sweep 1 energy=-73.91116759515667  maxlinkdim=50 maxerr=5.04E-10 time=0.101
After sweep 2 energy=-73.91506966117664  maxlinkdim=57 maxerr=9.11E-15 time=0.111
After sweep 3 energy=-73.91515452080588  maxlinkdim=57 maxerr=9.89E-15 time=0.111
After sweep 4 energy=-73.91523217590363  maxlinkdim=58 maxerr=8.67E-15 time=0.133
After sweep 5 energy=-73.9153127339381  maxlinkdim=57 maxerr=7.86E-15 time=0.113
After sweep 6 energy=-73.91539660212122  maxlinkdim=57 maxerr=7.40E-15 time=0.116
After sweep 7 energy=-73.91549144831691  maxlinkdim=57 maxerr=8.15E-15 time=0.111
After sweep 8 energy=-73.91558186574274  maxlinkdim=58 maxerr=8.96E-15 time=0.111
After sweep 9 energy=-73.91568038854744  maxlinkdim=58 maxerr=6.35E-15 time=0.111
After sweep 10 energy=-73.9157595850628  maxlinkdim=59 maxerr=8.24E-15 time=0.109


(-73.9157595850628, MPS
[1] ((dim=4|id=840|"Electron,Site,n=1") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1, (dim=4|id=242|"Link,l=1") <Out>
 1: QN(("Nf",5,-1),("Sz",1)) => 1
 2: QN(("Nf",6,-1),("Sz",0)) => 1
 3: QN(("Nf",6,-1),("Sz",2)) => 1
 4: QN(("Nf",7,-1),("Sz",1)) => 1)
[2] ((dim=16|id=599|"Link,l=2") <Out>
 1: QN(("Nf",3,-1),("Sz",1)) => 1
 2: QN(("Nf",4,-1),("Sz",0)) => 2
 3: QN(("Nf",4,-1),("Sz",2)) => 2
 4: QN(("Nf",5,-1),("Sz",-1)) => 1
 5: QN(("Nf",5,-1),("Sz",1)) => 4
 6: QN(("Nf",5,-1),("Sz",3)) => 1
 7: QN(("Nf",6,-1),("Sz",0)) => 2
 8: QN(("Nf",6,-1),("Sz",2)) => 2
 9: QN(("Nf",7,-1),("Sz",1)) => 1, (dim=4|id=354|"Electron,Site,n=2") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1, (dim=4|id=242|"Link,l=1") <In>
 1: QN(("Nf",5,-1),("Sz",1)) => 1
 2: QN(("Nf",6,-1),("Sz",0)) => 1

In [97]:
using LinearAlgebra

function Ep(rdm, L)
    println("trace (rho) = ", tr(rdm))
    println("ishermitian(rho) = ", ishermitian(rdm))
    lambdas = eigvals(rdm)
    S = 0
    println("eigenvals = ", lambdas)
    for lambda in lambdas
        lambda = real(lambda) 
        # S -= lambda * log2(lambda)
        # CRITICAL: Handle lambda <= 0 for log2
        if lambda > 1e-15 # A small threshold to avoid errors with log2
            S -= lambda * log2(lambda)
        end
    end 
    return S 
end

Ep (generic function with 1 method)

Resultado por ED: $\mathcal{L} = 0.7148148148148148$


In [98]:
rho_1 = build_1_particle_rdm(psi)

14×14 Matrix{ComplexF64}:
  8.90841e-6+0.0im          0.0+0.0im  …          0.0+0.0im
         0.0+0.0im   3.57177e-7+0.0im      2.32774e-5+0.0im
   0.0001066+0.0im          0.0+0.0im             0.0+0.0im
         0.0+0.0im   4.35507e-6+0.0im     0.000339946+0.0im
 0.000648598+0.0im          0.0+0.0im             0.0+0.0im
         0.0+0.0im   2.72974e-5+0.0im  …   0.00240912+0.0im
 -5.68656e-5+0.0im          0.0+0.0im             0.0+0.0im
         0.0+0.0im   2.92097e-5+0.0im      1.58892e-5+0.0im
  1.92128e-5+0.0im          0.0+0.0im             0.0+0.0im
         0.0+0.0im   -5.2007e-6+0.0im     -0.00127863+0.0im
 -9.44522e-5+0.0im          0.0+0.0im  …          0.0+0.0im
         0.0+0.0im  0.000166495+0.0im       0.0189945+0.0im
 -1.28781e-5+0.0im          0.0+0.0im             0.0+0.0im
         0.0+0.0im   2.32774e-5+0.0im      0.00265836+0.0im

In [99]:
E_p = Ep(rho_1, L) - log2(L)
println("E_p = ", E_p)

trace (rho) = 1.0000000000000007 + 0.0im
ishermitian(rho) = true
eigenvals = [1.0522980355540278e-8, 1.7000945621143245e-6, 7.948224692103305e-6, 1.394836099780942e-5, 1.4442753137502993e-5, 0.0009070491838730637, 0.0009077640859899641, 0.14193934003918798, 0.1419393428351171, 0.142848565273484, 0.14285366582498746, 0.14285385206409665, 0.14285524576062666, 0.14285712497626743]
E_p = 0.016428105273210036


In [90]:
  N = 8
  m = 4

  s = siteinds("Electron", N; conserve_qns=true)
  psi = random_mps(s, n -> isodd(n) ? "Up" : "Dn"; linkdims=m)
  
  Cuu = correlation_matrix(psi, "Cdagup", "Cup")

8×8 Matrix{Float64}:
  0.375793      0.0317322    -0.069603    …   0.000810962   0.00226663
  0.0317322     0.603177     -0.0187863       0.000218884   0.000611779
 -0.069603     -0.0187863     0.571713        0.00540164    0.0129137
 -0.00789479   -0.00213086   -0.0423659      -0.0605959    -0.0889724
  0.00416379    0.00112383    0.0181455      -0.0821161    -0.134697
 -0.00126828   -0.000342318  -0.00569325  …   0.0678847     0.122088
  0.000810962   0.000218884   0.00540164      0.679848      0.0923111
  0.00226663    0.000611779   0.0129137       0.0923111     0.6065

- trying to create a non-uniform mesh grid to compute the phase diagram.

In [ ]:
param1_segment1 = range(0.0, stop=0.9, length=10)
param1_segment2 = range(0.91, stop=1.09, length=40)
param1_segment3 = range(1.1, stop=2.0, length=10)

param1_values = vcat(collect(param1_segment1), collect(param1_segment2), collect(param1_segment3))

param2_values = collect(range(0.0, stop=1.0, length=50))

50-element Vector{Float64}:
 0.0
 0.02040816326530612
 0.04081632653061224
 0.061224489795918366
 0.08163265306122448
 0.10204081632653061
 0.12244897959183673
 0.14285714285714285
 0.16326530612244897
 0.1836734693877551
 ⋮
 0.8367346938775511
 0.8571428571428571
 0.8775510204081632
 0.8979591836734694
 0.9183673469387755
 0.9387755102040817
 0.9591836734693877
 0.9795918367346939
 1.0

In [75]:
function build_1_particle_rdm(psi) 
    L = length(psi)
    #=  
        N and not L since any site can have spin up or down
    =#
    rho_1 = zeros(ComplexF64, 2*L, 2*L) 

    Cupup = correlation_matrix(psi, "Cdagup", "Cup")
    Cdndn = correlation_matrix(psi, "Cdagdn", "Cdn")
    Cupdn = correlation_matrix(psi, "Cdagup", "Cdn")
    Cdnup = correlation_matrix(psi, "Cdagdn", "Cup")   

    for i in 1:L
        for j in 1:L
            # Blocks of the correlation matrix.
            i_up = 2 * (i-1) + 1 
            i_dn = 2 * (i-1) + 2
            j_up = 2 * (j-1) + 1
            j_dn = 2 * (j-1) + 2

            rho_1[i_up, j_up] = Cupup[i,j]
            rho_1[i_up, j_dn] = Cupdn[i,j]
            rho_1[i_dn, j_up] = Cdnup[i,j]
            rho_1[i_dn, j_dn] = Cdndn[i,j]
        end 
    end
    rho_1 = rho_1 / L
end

build_1_particle_rdm (generic function with 1 method)

In [53]:
N = 2
m = 4

s = siteinds("Electron", N; conserve_qns=true)

psi = productMPS(s, ["Up", "Dn"])

MPS
[1] ((dim=4|id=749|"Electron,Site,n=1") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1, (dim=1|id=240|"Link,l=1") <In>
 1: QN(("Nf",1,-1),("Sz",1)) => 1)
[2] ((dim=1|id=240|"Link,l=1") <Out>
 1: QN(("Nf",1,-1),("Sz",1)) => 1, (dim=4|id=753|"Electron,Site,n=2") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1)


In [54]:
rho_1 = build_1_particle_rdm(psi)

println(tr(rho_1))
println("is hermitian ? ", ishermitian(rho_1))

1.0 + 0.0im
is hermitian ? true


In [56]:
real(rho_1)

4×4 Matrix{Float64}:
 0.5  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.5

In [57]:
println(size(rho_1))

(4, 4)


In [ ]:
Cupup = correlation_matrix(psi, "Cdagup", "Cup")
Cdndn = correlation_matrix(psi, "Cdagdn", "Cdn")
Cupdn = correlation_matrix(psi, "Cdagup", "Cdn")
Cdnup = correlation_matrix(psi, "Cdagdn", "Cup")   

println("Cupup = ", Cupup)
println("Cupdn = ", Cupdn)
println("Cdnup = ", Cdnup)

println("Cdndn = ", Cdndn)

Cupup = [1.0 0.0; 0.0 0.0]
Cupdn = [0.0 0.0; 0.0 0.0]
Cdnup = [0.0 0.0; 0.0 0.0]
Cdndn = [0.0 0.0; 0.0 1.0]


In [60]:
E_p = Ep(rho_1, N) - log2(N)

println("E_p = ", E_p)

trace (rho) = 1.0 + 0.0im
ishermitian(rho) = true
eigenvals = [0.0, 0.0, 0.5, 0.5]
E_p = 0.0


2-rdm

In [ ]:
os = OpSum()
for i in 1:L 
    for j in 1:L 
        for k in 1:L 
            for l in 1:L 
            end
        end
    end
end

MethodError: MethodError: no method matching (ITensors.LazyApply.Applied{typeof(sum), Tuple{Array{ITensors.LazyApply.Applied{typeof(*), Tuple{C, Prod{Op}}, @NamedTuple{}}, 1}}, @NamedTuple{}} where C)(::Vector{Index{Vector{Pair{QN, Int64}}}})
The type `ITensors.LazyApply.Applied{typeof(sum), Tuple{Array{ITensors.LazyApply.Applied{typeof(*), Tuple{C, Prod{Op}}, @NamedTuple{}}, 1}}, @NamedTuple{}} where C` exists, but no method is defined for this combination of argument types when trying to construct it.

Closest candidates are:
  (ITensors.LazyApply.Applied{typeof(sum), Tuple{Array{ITensors.LazyApply.Applied{typeof(*), Tuple{C, Prod{Op}}, @NamedTuple{}}, 1}}, @NamedTuple{}} where C)()
   @ ITensors ~/.julia/packages/ITensors/6du4Z/src/lib/Ops/src/op.jl:184
